In [38]:
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

In [34]:
lct = pd.read_csv('lct_clinvar_result.txt', sep='\t')

lct_affect = lct[(lct['Variant type'] == 'single nucleotide variant') &
                 (lct['Condition(s)'].str.contains('lact', case=False, na=False)) &
                 (lct['dbSNP ID'].notna()) ]

mcm6 = pd.read_csv('mcm6_clinvar_result.txt', sep='\t')
mcm6_affect = mcm6[(mcm6['Variant type'] == 'single nucleotide variant') &
                   (mcm6['Condition(s)'].str.contains('lact', case=False, na=False)) &
                   (mcm6['dbSNP ID'].notna()) ]

lactose = pd.concat([lct_affect, mcm6_affect], ignore_index=True, sort=False)

lactose.to_csv('lactose.csv', sep=',', index=False)
lactose['dbSNP ID'].to_csv('lactose_id_only.txt', sep=',', index=False)

In [36]:
adh1a = pd.read_csv('adh1a_clinvar_result.txt', sep='\t')
adh1b = pd.read_csv('adh1b_clinvar_result.txt', sep='\t')
adh1c = pd.read_csv('adh1c_clinvar_result.txt', sep='\t')
adh4 = pd.read_csv('adh4_clinvar_result.txt', sep='\t')
adh5 = pd.read_csv('adh5_clinvar_result.txt', sep='\t')
adh6 = pd.read_csv('adh6_clinvar_result.txt', sep='\t')
aldh2 = pd.read_csv('aldh2_clinvar_result.txt', sep='\t')

alcohol_raw = pd.concat([aldh2,adh6,adh4,adh5,adh1a,adh1b,adh1c], ignore_index=True, sort=False)

alcohol = alcohol_raw[(alcohol_raw['Variant type'] == 'single nucleotide variant') &
                      (alcohol_raw['Condition(s)'].str.contains("alcohol|ethanol", case=False, na=False)) &
                      (alcohol_raw['dbSNP ID'].notna())]

alcohol.to_csv('alcohol.csv', sep=',', index=False)
alcohol['dbSNP ID'].to_csv('alcohol_id_only.txt', sep=',', index=False)

In [79]:
kz = pd.read_csv("../three_populations/kazakh_folder/kazakh_freq.frq", delim_whitespace=True)
asia = pd.read_csv("../three_populations/asian_folder/asian_freq.frq", delim_whitespace=True)
asia["MAF"] = pd.to_numeric(asia["MAF"], errors="coerce")

merged_asia = pd.merge(kz, asia, on = "SNP", suffixes=("_kz","_asia"))
flip_mask_asia = merged_asia['A1_kz'] != merged_asia['A1_asia']

merged_asia["minor_kz"] = (merged_asia["MAF_kz"] *
                           merged_asia["NCHROBS_kz"]).round().astype(int)
merged_asia["major_kz"] = (merged_asia["NCHROBS_kz"] -
                           merged_asia["minor_kz"])

merged_asia["minor_asia"] = (merged_asia["MAF_asia"] *
                             merged_asia["NCHROBS_asia"]).round().astype(int)
merged_asia["major_asia"] = (merged_asia["NCHROBS_asia"] -
                             merged_asia["minor_asia"])

merged_asia.loc[flip_mask_asia, ['minor_asia', 'major_asia']] = (
    merged_asia.loc[flip_mask_asia, ['major_asia', 'minor_asia']].values)
merged_asia.loc[flip_mask_asia, ['A1_asia', 'A2_asia']] = (
    merged_asia.loc[flip_mask_asia, ['A2_asia', 'A1_asia']].values)

merged_asia.loc[flip_mask_asia, 'MAF_asia'] = \
    (1 - merged_asia.loc[flip_mask_asia, 'MAF_asia'])

p_values = []
for _, row in merged_asia.iterrows():
    table = [[row["minor_kz"], row["major_kz"]],
             [row["minor_asia"], row["major_asia"]]]
    try:
        chi2, p, _, _ = chi2_contingency(table)
    except:
        p = 1.0
    p_values.append(p)

merged_asia["p_value"] = p_values

_, corrected_p, _, _ = multipletests(merged_asia["p_value"], method='bonferroni')
merged_asia["p_bonferroni"] = corrected_p
# MAF diff
merged_asia["MAF_diff"] = (merged_asia["MAF_kz"] - merged_asia["MAF_asia"]).abs()
top_SNPs_asia = merged_asia[merged_asia["p_bonferroni"] < 0.05].sort_values("p_bonferroni")

top_SNPs_asia[["SNP", "MAF_kz", "MAF_asia", "MAF_diff", "p_value", "p_bonferroni"]]

/var/folders/vz/rlwzk9b96rq1c7f4whbh21t00000gn/T/ipykernel_68269/3495575607.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  kz = pd.read_csv("../three_populations/kazakh_folder/kazakh_freq.frq", delim_whitespace=True)
/var/folders/vz/rlwzk9b96rq1c7f4whbh21t00000gn/T/ipykernel_68269/3495575607.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  asia = pd.read_csv("../three_populations/asian_folder/asian_freq.frq", delim_whitespace=True)


,SNP,MAF_kz,MAF_asia,MAF_diff,p_value,p_bonferroni
91526,rs12442525,0.2839,0.95041,0.66651,2.242763e-50,2.786363e-45
30079,rs2641726,0.1068,0.78510,0.67830,2.160552e-49,2.684227e-44
89581,rs2272587,0.3263,0.90496,0.57866,2.944157e-38,3.657762e-33
61088,rs10758700,0.2500,0.83060,0.58060,2.116943e-36,2.630048e-31
118360,rs709206,0.2521,0.82230,0.57020,2.953604e-35,3.669499e-30
...,...,...,...,...,...,...
18157,rs6923,0.3263,0.12810,0.19820,3.970812e-07,4.933257e-02
36927,rs34708521,0.1525,0.35950,0.20700,3.972881e-07,4.935828e-02
83789,rs7965570,0.3491,0.14460,0.20450,3.973648e-07,4.936781e-02
95933,rs2719712,0.3856,0.17360,0.21200,3.979562e-07,4.944129e-02


In [83]:
euro = pd.read_csv("../three_populations/euro_folder/euro_freq.frq", delim_whitespace=True)
euro["MAF"] = pd.to_numeric(euro["MAF"], errors="coerce")

merged_euro = pd.merge(kz, euro, on = "SNP", suffixes=("_kz","_euro"))
flip_mask_euro = merged_euro['A1_kz'] != merged_euro['A1_euro']

merged_euro["minor_kz"] = (merged_euro["MAF_kz"] *
                           merged_euro["NCHROBS_kz"]).round().astype(int)
merged_euro["major_kz"] = (merged_euro["NCHROBS_kz"] -
                           merged_euro["minor_kz"])

merged_euro["minor_euro"] = (merged_euro["MAF_euro"] *
                             merged_euro["NCHROBS_euro"]).round().astype(int)
merged_euro["major_euro"] = (merged_euro["NCHROBS_euro"] -
                             merged_euro["minor_euro"])

merged_euro.loc[flip_mask_euro, ['minor_euro', 'major_euro']] = (
    merged_euro.loc[flip_mask_euro, ['major_euro', 'minor_euro']].values)
merged_euro.loc[flip_mask_euro, ['A1_euro', 'A2_euro']] = (
    merged_euro.loc[flip_mask_euro, ['A2_euro', 'A1_euro']].values)

merged_euro.loc[flip_mask_euro, 'MAF_euro'] = (
        1 - merged_euro.loc[flip_mask_euro, 'MAF_euro'])

p_values = []
for _, row in merged_euro.iterrows():
    table = [[row["minor_kz"], row["major_kz"]],
             [row["minor_euro"], row["major_euro"]]]
    try:
        chi2, p, _, _ = chi2_contingency(table)
    except:
        p = 1.0
    p_values.append(p)

merged_euro["p_value"] = p_values

_, corrected_p, _, _ = multipletests(merged_euro["p_value"], method='bonferroni')
merged_euro["p_bonferroni"] = corrected_p
# MAF diff
merged_euro["MAF_diff"] = (merged_euro["MAF_kz"] - merged_euro["MAF_euro"]).abs()
top_SNPs_euro = merged_euro[merged_euro["p_bonferroni"] < 0.05].sort_values("p_bonferroni")

top_SNPs_euro[["SNP", "MAF_kz", "MAF_euro", "MAF_diff", "p_value", "p_bonferroni"]]

/var/folders/vz/rlwzk9b96rq1c7f4whbh21t00000gn/T/ipykernel_68269/3422973398.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  euro = pd.read_csv("../three_populations/euro_folder/euro_freq.frq", delim_whitespace=True)


,SNP,MAF_kz,MAF_euro,MAF_diff,p_value,p_bonferroni
64894,rs200385840,0.02119,0.84710,0.82591,3.298402e-73,4.097869e-68
99699,rs35229416,0.02586,0.68600,0.66014,7.462062e-50,9.270717e-45
36840,rs16891982,0.32910,0.94215,0.61305,1.416719e-43,1.760104e-38
95070,rs79486249,0.04741,0.64880,0.60139,4.850055e-42,6.025611e-37
36838,rs35407,0.38140,0.95868,0.57728,1.018921e-40,1.265888e-35
...,...,...,...,...,...,...
55695,rs2293909,0.38560,0.17360,0.21200,3.979562e-07,4.944129e-02
61278,rs12238629,0.38560,0.17360,0.21200,3.979562e-07,4.944129e-02
46511,rs72935743,0.38560,0.17360,0.21200,3.979562e-07,4.944129e-02
7176,rs1819347,0.30340,0.53720,0.23380,3.988202e-07,4.954863e-02


In [95]:
lactose_euro = top_SNPs_euro[top_SNPs_euro['SNP'].isin(lactose['dbSNP ID'])]
lactose_asia = top_SNPs_asia[top_SNPs_asia['SNP'].isin(lactose['dbSNP ID'])]

alcohol_euro = top_SNPs_euro[top_SNPs_euro['SNP'].isin(alcohol['dbSNP ID'])]
alcohol_asia = top_SNPs_asia[top_SNPs_asia['SNP'].isin(alcohol['dbSNP ID'])]

In [111]:
merged_euro[merged_euro['SNP'].isin(lactose['dbSNP ID'])]

,CHR_kz,SNP,A1_kz,A2_kz,MAF_kz,NCHROBS_kz,CHR_euro,A1_euro,A2_euro,MAF_euro,NCHROBS_euro,minor_kz,major_kz,minor_euro,major_euro,p_value,p_bonferroni,MAF_diff
17986,2,rs77631953,G,T,0.016950,236,2,G,T,0.004132,242,4,232,1,241,0.353703,1.0,0.012818
17987,2,rs1042712,C,G,0.275400,236,2,C,G,0.219000,242,65,171,53,189,0.185479,1.0,0.056400
17988,2,rs3213891,G,A,0.123900,234,2,G,A,0.033060,242,29,205,8,234,0.000414,1.0,0.090840
17989,2,rs2322659,T,C,0.487200,234,2,T,C,0.388400,242,114,120,94,148,0.037612,1.0,0.098800
17990,2,rs2304371,G,A,0.313600,236,2,G,A,0.247900,242,74,162,60,182,0.134860,1.0,0.065700
17991,2,rs3739022,A,G,0.135600,236,2,A,G,0.173600,242,32,204,42,200,0.307421,1.0,0.038000
17993,2,rs35093754,C,G,0.059320,236,2,C,G,0.020660,242,14,222,5,237,0.053742,1.0,0.038660
17994,2,rs6719488,T,G,0.449200,236,2,T,G,0.574400,242,106,130,139,103,0.008121,1.0,0.125200
17996,2,rs3754689,T,C,0.360200,236,2,T,C,0.227300,242,85,151,55,187,0.001991,1.0,0.132900
17997,2,rs2236783,A,G,0.370700,232,2,A,G,0.574400,242,86,146,139,103,0.000014,1.0,0.203700


In [112]:
merged_asia[merged_asia['SNP'].isin(lactose['dbSNP ID'])]

,CHR_kz,SNP,A1_kz,A2_kz,MAF_kz,NCHROBS_kz,CHR_asia,A1_asia,A2_asia,MAF_asia,NCHROBS_asia,minor_kz,major_kz,minor_asia,major_asia,p_value,p_bonferroni,MAF_diff
17986,2,rs77631953,G,T,0.016950,236,2,G,T,0.004132,242,4,232,1,241,0.353703,1.0,0.012818
17987,2,rs1042712,C,G,0.275400,236,2,C,G,0.206600,242,65,171,50,192,0.098393,1.0,0.068800
17988,2,rs3213891,G,A,0.123900,234,2,G,A,0.190100,242,29,205,46,196,0.063657,1.0,0.066200
17989,2,rs2322659,T,C,0.487200,234,2,T,C,0.504100,242,114,120,122,120,0.780905,1.0,0.016900
17990,2,rs2304371,G,A,0.313600,236,2,G,A,0.194200,242,74,162,47,195,0.003791,1.0,0.119400
17991,2,rs3739022,A,G,0.135600,236,2,A,G,0.223100,242,32,204,54,188,0.017681,1.0,0.087500
17993,2,rs35093754,C,G,0.059320,236,2,C,G,0.041320,242,14,222,10,232,0.489250,1.0,0.018000
17994,2,rs6719488,T,G,0.449200,236,2,T,G,0.458700,242,106,130,111,131,0.906665,1.0,0.009500
17996,2,rs3754689,T,C,0.360200,236,2,T,C,0.314000,242,85,151,76,166,0.332112,1.0,0.046200
17997,2,rs2236783,A,G,0.370700,232,2,A,G,0.425600,242,86,146,103,139,0.259690,1.0,0.054900


In [113]:
merged_asia[merged_asia['SNP'].isin(alcohol['dbSNP ID'])]

,CHR_kz,SNP,A1_kz,A2_kz,MAF_kz,NCHROBS_kz,CHR_asia,A1_asia,A2_asia,MAF_asia,NCHROBS_asia,minor_kz,major_kz,minor_asia,major_asia,p_value,p_bonferroni,MAF_diff
33855,4,rs1229984,T,C,0.2203,236,4,T,C,0.69010,242,52,184,167,75,1.722871e-24,2.140461e-19,0.46980
33860,4,rs698,C,T,0.1864,236,4,C,T,0.07025,242,44,192,17,225,2.431316e-04,1.000000e+00,0.11615


In [114]:
merged_euro[merged_euro['SNP'].isin(alcohol['dbSNP ID'])]

,CHR_kz,SNP,A1_kz,A2_kz,MAF_kz,NCHROBS_kz,CHR_euro,A1_euro,A2_euro,MAF_euro,NCHROBS_euro,minor_kz,major_kz,minor_euro,major_euro,p_value,p_bonferroni,MAF_diff
33855,4,rs1229984,T,C,0.2203,236,4,T,C,0.03306,242,52,184,8,234,1.533632e-09,0.000191,0.18724
33860,4,rs698,C,T,0.1864,236,4,C,T,0.37190,242,44,192,90,152,1.026638e-05,1.000000,0.18550


### Below i compared asia v euro

In [97]:
merged = pd.merge(euro, asia, on = "SNP", suffixes=("_euro","_asia"))
flip_mask = merged['A1_euro'] != merged['A1_asia']

merged["minor_euro"] = (merged["MAF_euro"] *
                           merged["NCHROBS_euro"]).round().astype(int)
merged["major_euro"] = (merged["NCHROBS_euro"] -
                           merged["minor_euro"])

merged["minor_asia"] = (merged["MAF_asia"] *
                             merged["NCHROBS_asia"]).round().astype(int)
merged["major_asia"] = (merged["NCHROBS_asia"] -
                             merged["minor_asia"])

merged.loc[flip_mask, ['minor_asia', 'major_asia']] = (
    merged.loc[flip_mask, ['major_asia', 'minor_asia']].values)
merged.loc[flip_mask, ['A1_asia', 'A2_asia']] = (
    merged.loc[flip_mask, ['A2_asia', 'A1_asia']].values)

merged.loc[flip_mask, 'MAF_asia'] = \
    (1 - merged.loc[flip_mask, 'MAF_asia'])

p_values = []
for _, row in merged.iterrows():
    table = [[row["minor_euro"], row["major_euro"]],
             [row["minor_asia"], row["major_asia"]]]
    try:
        chi2, p, _, _ = chi2_contingency(table)
    except:
        p = 1.0
    p_values.append(p)

merged["p_value"] = p_values

_, corrected_p, _, _ = multipletests(merged["p_value"], method='bonferroni')
merged["p_bonferroni"] = corrected_p
# MAF diff
merged["MAF_diff"] = (merged["MAF_euro"] - merged["MAF_asia"]).abs()
top_SNPs = merged[merged["p_bonferroni"] < 0.05].sort_values("p_bonferroni")

lactose_euro_asia = top_SNPs[top_SNPs['SNP'].isin(lactose['dbSNP ID'])]
alcohol_euro_asia = top_SNPs[top_SNPs['SNP'].isin(alcohol['dbSNP ID'])]

In [115]:
lactose_euro_asia

,CHR_euro,SNP,A1_euro,A2_euro,MAF_euro,NCHROBS_euro,CHR_asia,A1_asia,A2_asia,MAF_asia,NCHROBS_asia,minor_euro,major_euro,minor_asia,major_asia,p_value,p_bonferroni,MAF_diff
19923,2,rs3213891,G,A,0.03306,242,2,G,A,0.1901,242,8,234,46,196,9.199273e-08,0.012651,0.15704


In [116]:
alcohol_euro_asia

,CHR_euro,SNP,A1_euro,A2_euro,MAF_euro,NCHROBS_euro,CHR_asia,A1_asia,A2_asia,MAF_asia,NCHROBS_asia,minor_euro,major_euro,minor_asia,major_asia,p_value,p_bonferroni,MAF_diff
37603,4,rs1229984,T,C,0.03306,242,4,T,C,0.69010,242,8,234,167,75,1.606289e-50,2.208985e-45,0.65704
37608,4,rs698,C,T,0.37190,242,4,C,T,0.07025,242,90,152,17,225,3.104043e-15,4.268712e-10,0.30165
